In [2]:
pip install pandas sqlalchemy duckdb openpyxl jupysql

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.2 MB 2.6 MB/s eta 0:00:05
   --- ------------------------------------ 1.0/13.2 MB 2.6 MB/s eta 0:00:05
   --- ------------------------------------ 1.3/13.2 MB 2.3 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/13.2 MB 2.7 MB/s eta 0:00:05
   -------- ------------------------------- 2.9/13.2 MB 2.8 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/13.2 MB 2.8 MB/s eta 0:00:04
   ------------- -------------------------- 4.5/13.2 MB 3.1 MB/s eta 0:00:03
   --------------- ------------------------ 5.2/13.2 MB 3.3 MB/s eta 0:00:03
   ------------------ --------------------- 6.0/13.2 MB 3.4 MB/s eta 0:00:03
   --------------------- ------------------ 7.1/13.2 MB 3.5 MB/s eta 0:00:02
   ----------------------- ---------------- 7.9/13.2 MB 3.5 MB/s eta 0:00:02
   ----------

In [3]:
import duckdb
import pandas as pd

conn = duckdb.connect("retail_analytics.db")

df_raw = pd.DataFrame(
    {
        "transaction_id": ["T101", "T102", "T103", "T104", "T105", "T106"],
        "customer_id": ["C01", "C02", "C01", "C03", "C02", "C01"],
        "order_date": pd.to_datetime(
            [
                "2024-01-15",
                "2024-01-20",
                "2024-02-10",
                "2024-02-18",
                "2024-03-05",
                "2024-03-12",
            ]
        ),
        "product_category": [
            "Electronics",
            "Home Decor",
            "Electronics",
            "Apparel",
            "Home Decor",
            "Apparel",
        ],
        "quantity": [2, 1, 1, 3, 2, 4],
        "unit_price": [150.0, 45.0, 300.0, 25.0, 45.0, 30.0],
        "discount": [0.05, 0.0, 0.10, 0.0, 0.05, 0.0],
        "shipping_cost": [12.0, 5.0, 15.0, 7.0, 8.0, 10.0],
    }
)

conn.register("raw_sales", df_raw)

In [9]:
rfm_query = """
WITH customer_aggregates AS (
    SELECT
        customer_id,
        MAX(order_date) AS last_order_date,
        COUNT(DISTINCT transaction_id) AS total_orders,
        SUM((quantity * unit_price * (1 - discount)) + shipping_cost) AS total_spend
    FROM raw_sales
    GROUP BY customer_id
)
SELECT
    customer_id,
    DATE_DIFF('day', last_order_date, DATE '2024-04-01') AS recency_days,
    total_orders AS frequency,
    ROUND(total_spend, 2) AS monetary_value,
    NTILE(5) OVER (ORDER BY DATE_DIFF('day', last_order_date, DATE '2024-04-01') DESC) AS r_score,
    NTILE(5) OVER (ORDER BY total_orders ASC) AS f_score,
    NTILE(5) OVER (ORDER BY total_spend ASC) AS m_score
FROM customer_aggregates
ORDER BY monetary_value DESC;
"""

df_rfm = conn.execute(rfm_query).fetchdf()

In [10]:
category_query = """
SELECT
    product_category,
    STRFTIME(order_date, '%Y-%m') AS sale_month,
    SUM(quantity) AS units_sold,
    ROUND(SUM(quantity * unit_price * (1 - discount)), 2) AS net_revenue,
    ROUND(AVG(discount), 4) AS avg_discount_rate
FROM raw_sales
GROUP BY product_category, STRFTIME(order_date, '%Y-%m')
ORDER BY sale_month ASC, net_revenue DESC;
"""

df_category = conn.execute(category_query).fetchdf()

In [11]:
cohort_query = """
WITH first_orders AS (
    SELECT
        customer_id,
        DATE_TRUNC('month', MIN(order_date)) AS cohort_month
    FROM raw_sales
    GROUP BY customer_id
),
monthly_activity AS (
    SELECT
        s.customer_id,
        f.cohort_month,
        DATE_TRUNC('month', s.order_date) AS activity_month,
        (EXTRACT(YEAR FROM s.order_date) - EXTRACT(YEAR FROM f.cohort_month)) * 12 +
        (EXTRACT(MONTH FROM s.order_date) - EXTRACT(MONTH FROM f.cohort_month)) AS period_number
    FROM raw_sales s
    JOIN first_orders f ON s.customer_id = f.customer_id
)
SELECT
    STRFTIME(cohort_month, '%Y-%m') AS cohort,
    period_number,
    COUNT(DISTINCT customer_id) AS active_customers
FROM monthly_activity
GROUP BY cohort, period_number
ORDER BY cohort, period_number;
"""

df_cohort = conn.execute(cohort_query).fetchdf()

In [7]:
output_path = "retail_analysis_portfolio.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_raw.to_excel(writer, sheet_name="Raw_Transactions", index=False)
    df_rfm.to_excel(writer, sheet_name="RFM_Segmentation", index=False)
    df_category.to_excel(writer, sheet_name="Category_Monthly", index=False)
    df_cohort.to_excel(writer, sheet_name="Cohort_Data", index=False)

print(f"Data exported successfully to {output_path}")

Data exported successfully to retail_analysis_portfolio.xlsx


In [12]:
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

wb = openpyxl.load_workbook("retail_analysis_portfolio.xlsx")

header_fill = PatternFill(
    start_color="1F4E78", end_color="1F4E78", fill_type="solid"
)
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

for ws in wb.worksheets:
    ws.views.sheetView[0].showGridLines = True

    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)

        for cell in col:
            cell.border = thin_border
            if cell.row == 1:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal="center", vertical="center")
            else:
                cell.font = Font(name="Calibri", size=11)

            val_str = str(cell.value or "")
            if len(val_str) > max_len:
                max_len = len(val_str)

        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

    table_ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
    table_name = f"Tbl_{ws.title.replace(' ', '_')}"
    tab = Table(displayName=table_name, ref=table_ref)

    style = TableStyleInfo(
        name="TableStyleMedium9",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )
    tab.tableStyleInfo = style
    ws.add_table(tab)

ws_rfm = wb["RFM_Segmentation"]
for row in ws_rfm.iter_rows(min_row=2, min_col=4, max_col=4):
    for cell in row:
        cell.number_format = "$#,##0.00"

ws_cat = wb["Category_Monthly"]
for row in ws_cat.iter_rows(min_row=2, min_col=4, max_col=4):
    for cell in row:
        cell.number_format = "$#,##0.00"
for row in ws_cat.iter_rows(min_row=2, min_col=5, max_col=5):
    for cell in row:
        cell.number_format = "0.00%"

wb.save("retail_analysis_portfolio_formatted.xlsx")

In [13]:
import pandas as pd

excel_data = pd.read_excel(
    "retail_analysis_portfolio_formatted.xlsx", sheet_name=None
)

for sheet_name, df in excel_data.items():
    print(f"--- Sheet: {sheet_name} ({df.shape[0]} rows, {df.shape[1]} columns) ---")
    display(df.head())

--- Sheet: Raw_Transactions (6 rows, 8 columns) ---


,transaction_id,customer_id,order_date,product_category,quantity,unit_price,discount,shipping_cost
0,T101,C01,2024-01-15,Electronics,2,150,0.05,12
1,T102,C02,2024-01-20,Home Decor,1,45,0.00,5
2,T103,C01,2024-02-10,Electronics,1,300,0.10,15
3,T104,C03,2024-02-18,Apparel,3,25,0.00,7
4,T105,C02,2024-03-05,Home Decor,2,45,0.05,8


--- Sheet: RFM_Segmentation (3 rows, 7 columns) ---


,customer_id,recency_days,frequency,monetary_value,r_score,f_score,m_score
0,C01,20,3,712.0,3,3,3
1,C02,27,2,143.5,2,2,2
2,C03,43,1,82.0,1,1,1


--- Sheet: Category_Monthly (6 rows, 5 columns) ---


,product_category,sale_month,units_sold,net_revenue,avg_discount_rate
0,Electronics,2024-01,2,285.0,0.05
1,Home Decor,2024-01,1,45.0,0.00
2,Electronics,2024-02,1,270.0,0.10
3,Apparel,2024-02,3,75.0,0.00
4,Apparel,2024-03,4,120.0,0.00


--- Sheet: Cohort_Data (4 rows, 3 columns) ---


,cohort,period_number,active_customers
0,2024-01,0,2
1,2024-01,1,1
2,2024-01,2,2
3,2024-02,0,1


In [14]:
import os

print(os.getcwd())

C:\Users\DHARA
